**Модуль 3. Анатомия Dockerfile и магия кэширования слоёв**

### 3.1. От готовых образов к своим: зачем писать Dockerfile

В прошлом модуле мы запускали готовые образы (`python:3.11-slim`, `redis:7`, `hello-world`). Мы брали чужой «ящик», открывали его и использовали. Но в реальной жизни вам нужен **свой** ящик — со своим кодом, своими библиотеками, своими настройками.

**Вопрос:** Как Docker узнает, из чего собрать образ?  
**Ответ:** Мы пишем для него **рецепт** — текстовый файл с именем `Dockerfile` (без расширения, именно так, с большой буквы). В этом файле пошагово описано: «возьми такую-то основу, установи то-то, скопируй сюда код, запусти это».

Представьте, что вы печёте пиццу. Готовый образ — это пицца из магазина замороженная. А `Dockerfile` — это ваш собственный рецепт: «Возьми тесто, намажь соусом, положи сыр, запеки 15 минут при 200°». Docker читает этот рецепт и строго следует инструкциям.

### 3.2. Что такое Dockerfile: файл-рецепт

**Dockerfile** — это обычный текстовый файл, который лежит в корне вашего проекта. В нём на специальном языке записаны инструкции для сборки образа.

**Важнейшее правило:** Docker читает Dockerfile **сверху вниз**, строка за строкой. Каждая инструкция создаёт новый **слой** образа. Это не просто последовательность команд — это конвейер, где результат каждого шага передаётся следующему.

**Создайте рабочую папку:**

In [ ]:
mkdir ~/docker-module3
cd ~/docker-module3

Внутри создайте файл с именем `Dockerfile` (без `.txt`, без `.py`, просто `Dockerfile`).

### 3.3. Инструкции Dockerfile: разбор каждой детали

#### 3.3.1. `FROM` — выбор базы, фундамента

**Синтаксис:**

In [ ]:
FROM имя_образа:тег

**Что делает:** Указывает, какой готовый образ будет служить основой. Это как фундамент дома — всё строится поверх него.

**Аналогия:** Вы решили сделать бутерброд. `FROM` говорит: «Возьми в качестве основы ломтик бородинского хлеба». Всё остальное вы кладёте на этот ломтик.

**Примеры:**

In [ ]:
FROM python:3.11-slim
FROM ubuntu:22.04
FROM node:18-alpine

**Почему `python:3.11-slim`, а не просто `python`?**
- `python` без тега = `python:latest` — непредсказуемо, завтра может измениться.
- `3.11` — фиксирует мажорную и минорную версию.
- `slim` — облегчённый вариант. В нём нет компиляторов, документации, лишних утилит. Это уменьшает размер образа с ~900 МБ до ~120 МБ.

**Правило:** `FROM` всегда первая строка Dockerfile (кроме комментариев и специальных директив вроде `ARG`).

#### 3.3.2. `WORKDIR` — установка рабочей папки

**Синтаксис:**

In [ ]:
WORKDIR /путь/внутри/контейнера

**Что делает:** Создаёт папку внутри контейнера (если её нет) и делает её текущей. Все последующие команды (`COPY`, `RUN`, `CMD`) будут выполняться из этой папки.

**Аналогия:** Вы заходите в мастерскую. `WORKDIR /app` — это как повесить табличку на дверь: «Входите сюда, здесь вы будете работать». Все ваши инструменты и материалы окажутся в этой комнате.

**Пример:**

In [ ]:
WORKDIR /app

**Важно:** Всегда используйте `WORKDIR` вместо `cd` внутри `RUN`. Почему — поймём позже, когда разберём слои.

#### 3.3.3. `COPY` — копирование файлов с хоста в образ

**Синтаксис:**

In [ ]:
COPY источник_на_хосте назначение_в_контейнере

**Что делает:** Берёт файлы или папки с вашего компьютера (хоста) и копирует их внутрь образа.

**Аналогия:** Вы приносите в мастерскую свои чертежи и кладёте их на стол. `COPY` — это именно кладёт файлы внутрь «ящика», они становятся частью образа.

**Примеры:**

In [ ]:
COPY requirements.txt .
COPY ./src /app/src
COPY . .

**Разбор `COPY . .`:**
- Первый `.` — текущая папка на хосте (где лежит Dockerfile).
- Второй `.` — текущая папка внутри контейнера (куда копировать, обычно это `WORKDIR`).

**Важный нюанс:** `COPY` копирует файлы **во время сборки образа**, не во время запуска контейнера. Если вы изменили файл на хосте после сборки — в контейнере останется старая версия, пока вы не пересоберёте образ.

#### 3.3.4. `RUN` — выполнение команд во время сборки

**Синтаксис:**

In [ ]:
RUN команда

**Что делает:** Выполняет команду внутри контейнера **при сборке образа**. Результат команды (установленные пакеты, созданные файлы) сохраняется в образе.

**Аналогия:** Это этап подготовки мастерской. Вы прикручиваете к стенам полки, устанавливаете верстак, подключаёте электричество. Всё это делается **до** того, как в мастерскую начнут приходить заказы.

**Примеры:**

In [ ]:
RUN pip install --no-cache-dir -r requirements.txt
RUN apt-get update && apt-get install -y gcc
RUN mkdir -p /app/logs

**Почему `--no-cache-dir` в `pip install`?**  
По умолчанию `pip` сохраняет скачанные пакеты в кэш (`~/.cache/pip`), чтобы при повторной установке не качать заново. Но внутри Docker-образа этот кэш бесполезен — образ собирается один раз и распространяется. Кэш только раздувает размер образа. `--no-cache-dir` говорит pip: «Скачай и сразу удали временные файлы».

#### 3.3.5. `ENV` — переменные окружения

**Синтаксис:**

In [ ]:
ENV ИМЯ_ПЕРЕМЕННОЙ значение

**Что делает:** Устанавливает переменную окружения, доступную программам внутри контейнера.

**Аналогия:** Это как повесить в мастерской объявление: «Все инструменты после работы класть на верхнюю полку». Это правило видят все, кто входит в мастерскую.

**Примеры:**

In [ ]:
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV APP_ENV=production

**Зачем эти переменные:**
- `PYTHONDONTWRITEBYTECODE=1` — Python не будет создавать файлы `.pyc` (скомпилированный байт-код). Это мусор внутри контейнера.
- `PYTHONUNBUFFERED=1` — Python не будет буферизовать вывод. Логи сразу появляются на экране, а не ждут, пока накопится буфер.

#### 3.3.6. `EXPOSE` — документирование порта

**Синтаксис:**

In [ ]:
EXPOSE номер_порта

**Что делает:** Документирует, что приложение внутри контейнера слушает на этом порту. Это **не открывает** порт автоматически — это просто подсказка для человека, читающего Dockerfile.

**Аналогия:** Вы вешаете на дверь мастерской табличку: «Вход со двора». Это не делает дверь автоматически открытой — просто сообщает, где искать вход.

**Пример:**

In [ ]:
EXPOSE 8000

Чтобы порт реально был доступен с хоста, при запуске используется флаг `-p`, как мы делали в Модуле 2.

#### 3.3.7. `CMD` — команда по умолчанию при запуске

**Синтаксис:**

In [ ]:
CMD ["исполняемый_файл", "аргумент1", "аргумент2"]

(рекомендуемый формат — JSON-массив)

**Что делает:** Определяет, какая команда выполнится, когда контейнер **запускается**. Это главный процесс контейнера. Когда он завершается — контейнер останавливается.

**Аналогия:** Это инструкция на двери мастерской: «Когда откроете дверь, сразу включите свет и начните работу над заказом №5». Это действие, которое происходит при **открытии** ящика, а не при его **изготовлении**.

**Пример:**

In [ ]:
CMD ["python", "main.py"]
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

**Важно:** В Dockerfile может быть только **одна** инструкция `CMD`. Если их несколько — сработает последняя.

#### 3.3.8. `ENTRYPOINT` — неизменяемая точка входа

**Синтаксис:**

In [ ]:
ENTRYPOINT ["исполняемый_файл"]

**Что делает:** Похоже на `CMD`, но более жёсткое. Определяет исполняемый файл, который **всегда** будет запущен. Аргументы из `CMD` дополняют `ENTRYPOINT`.

**Аналогия:** `ENTRYPOINT` — это заводская пломба на механизме: «Этот ящик всегда запускает Python». `CMD` — это параметры, которые вы передаёте Python по умолчанию.

**Пример:**

In [ ]:
ENTRYPOINT ["python"]
CMD ["main.py"]

Если запустить контейнер без аргументов — выполнится `python main.py`.  
Если запустить с аргументом `docker run my_image script.py` — выполнится `python script.py`, а `main.py` из `CMD` будет проигнорирован.

**Для новичка:** Пока используйте только `CMD`. `ENTRYPOINT` нужен для более сложных сценариев.

### 3.4. Слои (Layers): фундаментальная концепция Docker

Теперь мы подходим к самому важному. Без понимания слоёв невозможно писать эффективные Dockerfile.

#### 3.4.1. Что такое слой?

**Слой** — это неизменяемый (read-only) набор изменений файловой системы. Каждая инструкция в Dockerfile (`FROM`, `RUN`, `COPY`, `ENV`) создаёт один или несколько слоёв.

**Аналогия: лазанья или слоёный торт**

Представьте, что вы собираете слоёный торт:
1. Первый слой — бисквит (`FROM`).
2. Второй слой — крем (`RUN apt-get install...`).
3. Третий слой — фрукты (`COPY requirements.txt...`).
4. Четвёртый слой — ещё крем (`RUN pip install...`).
5. Пятый слой — верхняя глазурь (`COPY . .`).
6. Шестой слой — вишенка сверху (`CMD`).

Каждый слой лежит поверх предыдущего. Вместе они образуют финальный торт — ваш образ.

**Ключевое свойство:** Слои **неизменяемы**. Когда вы добавляете новый слой, вы не редактируете предыдущие. Вы просто кладёте сверху новый слой, который «перекрывает» или «дополняет» то, что было ниже.

#### 3.4.2. Union File System: как слои склеиваются

Docker использует технологию **UnionFS** (конкретно OverlayFS). Она позволяет «накладывать» слои друг на друга так, что программа внутри контейнера видит единую файловую систему.

**Аналогия: прозрачные листы на проекторе**

Представьте старый проектор с прозрачными плёнками:
- На первой плёнке нарисована карта мира (`FROM`).
- На второй плёнке нарисованы границы стран (`RUN apt-get...`).
- На третьей — города (`COPY`).

Когда вы кладёте все плёнки друг на друга и включаете проектор — вы видите единую картинку. Но каждая плёнка остаётся отдельной и неизменной. Если убрать вторую плёнку — границы исчезнут, но карта мира и города останутся.

#### 3.4.3. Слои на практике: где они живут

Когда Docker собирает образ, он показывает каждый шаг:

In [ ]:
Step 1/6 : FROM python:3.11-slim
 ---> a1b2c3d4e5f6
Step 2/6 : WORKDIR /app
 ---> Running in 7f8g9h0i1j2
 ---> k3l4m5n6o7p8
Step 3/6 : COPY requirements.txt .
 ---> q9r0s1t2u3v4
...

Каждый `Step` — это слой. Строка `-> a1b2c3d4e5f6` — это **хеш слоя** (уникальный идентификатор). Docker использует эти хеши для кэширования.

### 3.5. Кэширование слоёв: магия Docker

Это самая мощная оптимизация Docker. Поняв её, вы сможете ускорить сборку образа в 10-50 раз.

#### 3.5.1. Как работает кэш?

Docker запоминает хеш каждого собранного слоя. При следующей сборке он сравнивает: «А изменилась ли инструкция или файлы, которые она использует?»

- **Если НЕТ** — Docker берёт готовый слой из кэша. Он не выполняет команду заново, а просто подставляет уже готовый результат. Это мгновенно.
- **Если ДА** — Docker выполняет инструкцию заново, создаёт новый слой. И **все последующие слои** тоже пересобираются, потому что они могли зависеть от изменившегося.

**Аналогия: сборка мебели из IKEA**

Представьте, что вы собираете шкаф по инструкции:
1. Прикрутить ножки (слой 1).
2. Прикрутить заднюю стенку (слой 2).
3. Установить полки (слой 3).
4. Прикрутить дверцы (слой 4).
5. Повесить ручки (слой 5).

Вы собрали шкаф. Потом решили: «Хочу другие ручки». Вы меняете только шаг 5. Шаги 1-4 не трогаете — они уже сделаны. Это и есть кэширование.

Но если вы решили: «Хочу другую заднюю стенку» — вам придётся разобрать шаги 3, 4, 5 и собрать заново, потому что полки и дверцы стоят на задней стенке.

#### 3.5.2. Правило кэширования: «Один изменившийся слой ломает всё, что ниже»

Dockerfile читается сверху вниз. Если слой на шаге 3 изменился — слои 4, 5, 6 и т.д. **гарантированно пересобираются**, даже если их инструкции не менялись.

**Почему?** Потому что каждый следующий слой — это изменение файловой системы, которая получилась на предыдущем шаге. Если предыдущий шаг дал другой результат — всё, что строится поверх него, может быть другим.

### 3.6. Критически важный порядок инструкций: защита кэша

Теперь мы подходим к **золотому правилу** написания Dockerfile, которое отличает новичка от профессионала.

#### 3.6.1. Проблема: часто меняющийся код

В типичном Python-проекте:
- `requirements.txt` меняется редко (раз в неделю или реже).
- Исходный код (`*.py`) меняется постоянно (каждые несколько минут при разработке).

#### 3.6.2. Плохой Dockerfile (кэш постоянно ломается)

In [ ]:
FROM python:3.11-slim

WORKDIR /app

# Копируем ВЕСЬ проект сразу
COPY . .

# Устанавливаем зависимости
RUN pip install --no-cache-dir -r requirements.txt

CMD ["python", "main.py"]

**Что происходит при каждом изменении кода:**

1. Вы меняете одну строку в `main.py`.
2. Собираете образ: `docker build -t my_app .`
3. Docker видит: инструкция `COPY . .` изменилась (файлы на хосте другие).
4. Слой `COPY . .` пересобирается.
5. Все слои ниже тоже пересобираются, включая `RUN pip install...`.
6. Docker заново качает все пакеты из интернета. Это занимает 2-5 минут.
7. Вы ждёте. Каждый. Раз. Когда меняете код.

**Это катастрофа для разработки.**

#### 3.6.3. Хороший Dockerfile (кэш защищён)

In [ ]:
FROM python:3.11-slim

WORKDIR /app

# Сначала копируем ТОЛЬКО requirements.txt
COPY requirements.txt .

# Устанавливаем зависимости
RUN pip install --no-cache-dir -r requirements.txt

# Теперь копируем весь остальной код
COPY . .

CMD ["python", "main.py"]

**Что происходит теперь:**

1. Вы меняете `main.py`.
2. Docker проверяет `COPY requirements.txt .` — файл не изменился. **Берёт из кэша.**
3. Docker проверяет `RUN pip install...` — предыдущий слой не изменился, инструкция та же. **Берёт из кэша.** Установка пакетов не происходит! Мгновенно.
4. Docker доходит до `COPY . .` — файлы изменились. Пересобирает только этот слой и последующие.
5. Сборка занимает 2-5 секунд вместо минут.

**Почему это работает?**  
Потому что `requirements.txt` копируется отдельно, до основного кода. Docker видит: этот файл не менялся -> слой с установкой зависимостей тоже не менялся -> берём готовый слой из кэша.

#### 3.6.4. Золотое правило порядка инструкций

> **Располагайте инструкции от наименее изменяемых к наиболее изменяемым.**

Идеальный порядок сверху вниз:
1. `FROM` (меняется редко — при обновлении базового образа).
2. `WORKDIR`, `ENV` (почти не меняются).
3. `COPY` файлов зависимостей (`requirements.txt`, `package.json`, `Cargo.toml`).
4. `RUN` установки зависимостей (`pip install`, `npm install`).
5. `COPY` всего остального кода (меняется постоянно).
6. `CMD` / `ENTRYPOINT` (меняется иногда).

### 3.7. Практика: пишем Dockerfile для FastAPI-приложения

#### Шаг 1. Создаём структуру проекта

В папке `~/docker-module3` создайте файлы:

In [ ]:
# Создаём файлы
touch main.py requirements.txt Dockerfile

**`main.py`:**

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="ML Prediction API")

@app.get("/")
def read_root():
    return {"message": "Hello from Docker!"}

@app.get("/health")
def health_check():
    return {"status": "healthy"}

@app.get("/predict")
def predict(x: float = 1.0, y: float = 2.0):
    # Условная ML-модель: простое предсказание
    result = x * 2.5 + y * 1.8 + 0.5
    return {
        "input": {"x": x, "y": y},
        "prediction": result,
        "model_version": "1.0.0"
    }

**`requirements.txt`:**

In [ ]:
fastapi==0.111.0
uvicorn[standard]==0.30.0

#### Шаг 2. Пишем первый, неоптимальный Dockerfile

Создайте файл `Dockerfile.bad`:

In [ ]:
FROM python:3.11

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Соберите и замерьте время:

In [ ]:
time docker build -f Dockerfile.bad -t my_app_bad .

In [ ]:
Days              : 0
Hours             : 0
Minutes           : 0
Seconds           : 42
Milliseconds      : 489
Ticks             : 424899980
TotalDays         : 0,000491782384259259
TotalHours        : 0,0118027772222222
TotalMinutes      : 0,708166633333333
TotalSeconds      : 42,489998
TotalMilliseconds : 42489,998

**Что плохо:**
- `FROM python:3.11` — полный образ, ~900 МБ. `slim` был бы лучше.
- `COPY . .` до `RUN pip install` — кэш ломается при любом изменении кода.
- `pip install` без `--no-cache-dir` — мусор в образе.
- Нет `EXPOSE` (хотя это не критично, но непрофессионально).

#### Шаг 3. Пишем оптимальный Dockerfile

Создайте файл `Dockerfile`:

In [ ]:
# Используем облегчённый образ фиксированной версии
FROM python:3.11-slim

# Отключаем создание .pyc и буферизацию вывода
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

# Устанавливаем рабочую директорию
WORKDIR /app

# СНАЧАЛА копируем только requirements.txt
# Это защищает кэш слоя с установкой зависимостей
COPY requirements.txt .

# Устанавливаем зависимости
# --no-cache-dir не оставляет мусор в образе
RUN pip install --no-cache-dir -r requirements.txt

# ТЕПЕРЬ копируем весь исходный код
# Этот слой будет пересобираться при каждом изменении кода,
# но слой с pip install останется в кэше
COPY . .

# Документируем порт (не открывает, а просто информирует)
EXPOSE 8000

# Команда запуска
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

#### Шаг 4. Собираем и замеряем

In [ ]:
time docker build -t my_app .

In [ ]:
Days              : 0
Hours             : 0
Minutes           : 0
Seconds           : 17
Milliseconds      : 38
Ticks             : 170389958
TotalDays         : 0,000197210599537037
TotalHours        : 0,00473305438888889
TotalMinutes      : 0,283983263333333
TotalSeconds      : 17,0389958
TotalMilliseconds : 17038,9958

Запишите время. Теперь измените `main.py` (например, измените `"model_version": "1.0.0"` на `"1.0.1"`) и пересоберите:

In [ ]:
time docker build -t my_app .

In [ ]:
Days              : 0
Hours             : 0
Minutes           : 0
Seconds           : 0
Milliseconds      : 964
Ticks             : 9646777
TotalDays         : 1,11652511574074E-05
TotalHours        : 0,000267966027777778
TotalMinutes      : 0,0160779616666667
TotalSeconds      : 0,9646777
TotalMilliseconds : 964,6777

**Что вы увидите:**

In [ ]:
Step 1/8 : FROM python:3.11-slim
 ---> Using cache
Step 2/8 : ENV PYTHONDONTWRITEBYTECODE=1
 ---> Using cache
Step 3/8 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
Step 4/8 : WORKDIR /app
 ---> Using cache
Step 5/8 : COPY requirements.txt .
 ---> Using cache
Step 6/8 : RUN pip install --no-cache-dir -r requirements.txt
 ---> Using cache
Step 7/8 : COPY . .
 ---> 新ый хеш (пересобрано)
Step 8/8 : CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
 ---> 新ый хеш (пересобрано)

Шаги 1-6 взяты из кэша **мгновенно**. Пересобрались только шаги 7-8. Сборка заняла 1-3 секунды вместо минут.

#### Шаг 5. Запускаем и проверяем

In [ ]:
docker run -d --name my_api -p 8000:8000 my_app

Проверьте в браузере:
- `http://localhost:8000/` -> `{"message":"Hello from Docker!"}`
- `http://localhost:8000/health` -> `{"status":"healthy"}`
- `http://localhost:8000/predict?x=10&y=5` -> предсказание

Посмотрите логи:

In [ ]:
docker logs -f my_api

Остановите:

In [ ]:
docker stop my_api
docker rm my_api

### 3.8. Сравнение: плохой vs хороший Dockerfile

| Аспект | `Dockerfile.bad` | `Dockerfile` (оптимальный) |
|--------|------------------|---------------------------|
| **Размер базы** | ~900 МБ (`python:3.11`) | ~120 МБ (`python:3.11-slim`) |
| **Кэш при изменении кода** | Ломается, `pip install` перезапускается | Защищён, `pip install` из кэша |
| **Время пересборки** | 2-5 минут | 1-3 секунды |
| **Мусор в образе** | Кэш pip остаётся | `--no-cache-dir` убирает |
| **Профессиональность** | Новичок | Продакшен-уровень |

### 3.9. Дополнительная практика: эксперименты с кэшем

#### Эксперимент 1. Убедитесь, что кэш работает

In [ ]:
# Первая сборка (холодный старт)
docker build --no-cache -t my_app_cold .
# Заметьте время

# Измените main.py
echo "# comment" >> main.py

# Вторая сборка (должна использовать кэш для pip)
docker build -t my_app_warm .
# Заметьте время — оно должно быть на порядок меньше

#### Эксперимент 2. Сломайте кэш намеренно

Измените `requirements.txt`, добавив новую строку:

In [ ]:
requests==2.32.0

Пересоберите:

In [ ]:
docker build -t my_app_broken_cache .

Теперь Docker увидит, что `requirements.txt` изменился (шаг 5), сломает кэш, и выполнит `RUN pip install` заново (шаг 6). Это правильно — ведь зависимости изменились.

### 3.10. Итоги модуля: чек-лист

- [ ] Понимаю, что `Dockerfile` — это рецепт сборки образа.
- [ ] Знаю назначение `FROM`, `WORKDIR`, `COPY`, `RUN`, `ENV`, `EXPOSE`, `CMD`.
- [ ] Понимаю, что каждая инструкция создаёт **слой** образа.
- [ ] Знаю, что слои **неизменяемы** и накладываются друг на друга.
- [ ] Понимаю механизм **кэширования**: Docker повторно использует неизменившиеся слои.
- [ ] Знаю **золотое правило**: сначала копировать файлы зависимостей, потом устанавливать их, потом копировать код.
- [ ] Умею писать оптимальный Dockerfile для Python-приложения.
- [ ] Умею замерять время сборки и проверять использование кэша.
- [ ] Понимаю, зачем нужны `PYTHONDONTWRITEBYTECODE` и `PYTHONUNBUFFERED`.
- [ ] Умею запускать собранный образ и проверять работу API внутри контейнера.

**В следующем модуле** мы научимся **многоэтапной сборке** (multi-stage builds) — технике, которая позволяет собирать тяжёлые ML-зависимости (вроде `numpy`, `pandas`, `scikit-learn`, требующих компиляторы C) и при этом получать лёгкий финальный образ без лишнего мусора.